In [15]:
# ==============================================================================
# 1. INSTALL ZERO-COST SYSTEM & FORENSIC LIBRARIES
# ==============================================================================
!apt-get -qq update && apt-get -qq install -y exiftool ffmpeg

!pip install -q \
    opencv-python-headless \
    scenedetect[opencv] \
    numpy \
    scipy \
    openai-whisper \
    tabulate \
    Pillow

import os
import json
import subprocess
import cv2
import numpy as np
import scipy.fftpack as fftpack
import whisper
import torch
from google.colab import files
from tabulate import tabulate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[+] Operational Hardware: {DEVICE.upper()}")

# ==============================================================================
# 2. MEDIA INGESTION
# ==============================================================================
print("\n" + "=" * 60)
print("STAGE 1: MEDIA INGESTION")
print("=" * 60)
uploaded = files.upload()
if not uploaded:
    raise ValueError("No video file selected. Run cell again to upload.")

VIDEO_PATH = list(uploaded.keys())[0]
print(f"[+] Processing Target: {VIDEO_PATH}")

# ==============================================================================
# 3. CONTAINER & ENCODER FORENSICS (EXIFTOOL)
# ==============================================================================
def audit_container_internals(file_path):
    print("\n[+] Stage 2: Auditing deep container headers...")
    cmd = ["exiftool", "-j", file_path]
    res = subprocess.run(cmd, capture_output=True, text=True)
    try:
        exif = json.loads(res.stdout)[0]
    except Exception:
        exif = {}

    encoder = exif.get("Encoder", exif.get("CompressorName", "Unknown"))
    software = exif.get("Software", "None")
    make = exif.get("Make", "None")
    model = exif.get("Model", "None")

    # Flag indicators
    anomalies = []
    if make == "None" and model == "None":
        anomalies.append("Missing physical sensor/camera hardware profile")
    if any(synth in str(encoder).lower() for synth in ["lavf", "ffmpeg", "isom"]):
        anomalies.append("Synthetic/software muxer signature (Lavf/FFmpeg pipeline)")

    return {
        "Make/Model": f"{make} {model}".strip(),
        "Encoder": encoder,
        "Software": software,
        "FPS": exif.get("VideoFrameRate", "N/A"),
        "Duration": exif.get("Duration", "N/A"),
        "Anomalies": anomalies
    }

container_data = audit_container_internals(VIDEO_PATH)

# ==============================================================================
# 4. FREQUENCY DOMAIN (2D FFT) ARTIFACT ANALYSIS
# ==============================================================================
def analyze_fft_spectrum(frame_gray):
    """
    Computes 2D Fast Fourier Transform to find high-frequency checkerboard
    artifacts typical of generative diffusion/GAN upscalers.
    """
    # Compute 2D FFT and shift zero-frequency component to center
    f = np.fft.fft2(frame_gray)
    fshift = np.fft.fftshift(f)
    magnitude_spectrum = 20 * np.log(np.abs(fshift) + 1e-9)

    # Measure ratio of extreme high-frequency band vs low/mid frequencies
    h, w = frame_gray.shape
    cy, cx = h // 2, w // 2
    r_inner = min(h, w) // 8
    r_outer = min(h, w) // 4

    y, x = np.ogrid[:h, :w]
    mask_inner = (x - cx)**2 + (y - cy)**2 <= r_inner**2
    mask_outer = ((x - cx)**2 + (y - cy)**2 <= r_outer**2) & ~mask_inner

    high_freq_ratio = np.mean(magnitude_spectrum[mask_outer]) / (np.mean(magnitude_spectrum[mask_inner]) + 1e-6)
    return float(high_freq_ratio)

# ==============================================================================
# 5. TEMPORAL CONTINUITY & DENSE OPTICAL FLOW WARPING
# ==============================================================================
def inspect_temporal_and_motion(file_path, sample_limit=120):
    print("\n[+] Stage 3: Measuring temporal coherence & motion field divergence...")
    cap = cv2.VideoCapture(file_path)

    prev_gray = None
    flow_divergences = []
    fft_scores = []
    motion_magnitudes = []

    frame_idx = 0
    while cap.isOpened() and frame_idx < sample_limit:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 1. Spatial FFT analysis
        fft_ratio = analyze_fft_spectrum(gray)
        fft_scores.append(fft_ratio)

        # 2. Temporal Farneback dense optical flow
        if prev_gray is not None:
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, gray, None,
                pyr_scale=0.5, levels=3, winsize=15,
                iterations=3, poly_n=5, poly_sigma=1.2, flags=0
            )
            mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

            # Real camera footage has smooth, structured vector flow.
            # Generative video creates chaotic acceleration/warping vectors.
            flow_variance = np.var(mag)
            flow_divergences.append(flow_variance)
            motion_magnitudes.append(np.mean(mag))

        prev_gray = gray
        frame_idx += 1

    cap.release()

    avg_fft = float(np.mean(fft_scores)) if fft_scores else 0.0
    mean_flow_variance = float(np.mean(flow_divergences)) if flow_divergences else 0.0
    mean_motion = float(np.mean(motion_magnitudes)) if motion_magnitudes else 0.0

    return {
        "frames_analyzed": frame_idx,
        "avg_fft_high_freq_ratio": avg_fft,
        "mean_flow_variance": mean_flow_variance,
        "mean_motion_intensity": mean_motion
    }

motion_data = inspect_temporal_and_motion(VIDEO_PATH)

# ==============================================================================
# 6. AUDIO INTEGRITY & DIALOGUE CHECK (WHISPER - ZERO COST)
# ==============================================================================
def audit_audio_track(file_path):
    print("\n[+] Stage 4: Transcribing speech & evaluating audio channel...")
    whisper_model = whisper.load_model("base", device=DEVICE)
    res = whisper_model.transcribe(file_path, task="transcribe")
    return {
        "lang": res.get("language", "unknown").upper(),
        "transcript": res.get("text", "").strip()
    }

audio_data = audit_audio_track(VIDEO_PATH)

# ==============================================================================
# 7. FORENSIC SYNTHESIS & EXECUTIVE REPORT
# ==============================================================================
print("\n" + "=" * 76)
print("             PHYSICS & TEMPORAL VIDEO AUTHENTICATION REPORT")
print("=" * 76)

# Section 1: Container Audit
print("\n[1] HARDWARE & CONTAINER PROVENANCE")
print("-" * 60)
header_table = [
    ["Camera Make/Model", container_data["Make/Model"]],
    ["Muxer / Encoder", container_data["Encoder"]],
    ["Frame Rate", container_data["FPS"]],
    ["Duration", container_data["Duration"]],
    ["Identified Red Flags", "\n".join(container_data["Anomalies"]) if container_data["Anomalies"] else "None"]
]
print(tabulate(header_table, headers=["Parameter", "Audit Result"], tablefmt="fancy_grid"))

# Section 2: Motion & Frequency Physical Metrics
print("\n[2] PHYSICS-BASED TEMPORAL & SPECTRAL METRICS")
print("-" * 60)

# Decision logic based on physical properties
fft_val = motion_data["avg_fft_high_freq_ratio"]
flow_var = motion_data["mean_flow_variance"]

# High flow variance relative to overall motion indicates morphing/fluid drift
drift_ratio = flow_var / (motion_data["mean_motion_intensity"] + 1e-4)

metric_table = [
    ["Frames Sampled", motion_data["frames_analyzed"]],
    ["Spectral High-Frequency Ratio (FFT)", f"{fft_val:.3f} (Normal: 0.60 - 1.15)"],
    ["Dense Motion Flow Variance", f"{flow_var:.2f}"],
    ["Temporal Deformation / Drift Index", f"{drift_ratio:.2f} (Suspicious if > 15.0)"]
]
print(tabulate(metric_table, headers=["Forensic Metric", "Observed Value"], tablefmt="fancy_grid"))

# Formulate Verdict
is_synthetic = False
reasons = []

if drift_ratio > 15.0 and motion_data["mean_motion_intensity"] > 0.5:
    is_synthetic = True
    reasons.append("High non-rigid motion drift (unnatural pixel warping across consecutive frames)")

if fft_val > 1.25 or fft_val < 0.50:
    is_synthetic = True
    reasons.append("Anomalous frequency spectrum decay (typical of generative diffusion latents)")

if container_data["Anomalies"]:
    reasons.append("Complete absence of physical sensor noise and camera hardware headers")

print("\n[3] FORENSIC CONCLUSION")
print("-" * 60)
if is_synthetic:
    print("VERDICT: ⚠️ HIGH PROBABILITY OF GENERATIVE AI / SYNTHETIC VIDEO")
    print("Primary Indicators:")
    for r in reasons:
        print(f"  • {r}")
else:
    print("VERDICT: ✅ CONSISTENT WITH REAL-WORLD OPTICAL RECORDING")
    print("Physical metrics match coherent camera sensor motion and natural frequency decay.")

# Section 4: Spoken Track
print("\n[4] SPOKEN AUDIO CONTEXT")
print("-" * 60)
print(f"Language : {audio_data['lang']}")
print(f"Content  : {audio_data['transcript'] or '[No dialogue detected - Ambient/Muted]'}")
print("=" * 76)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[+] Operational Hardware: CUDA

STAGE 1: MEDIA INGESTION


Saving tester q.mp4 to tester q (3).mp4
[+] Processing Target: tester q (3).mp4

[+] Stage 2: Auditing deep container headers...

[+] Stage 3: Measuring temporal coherence & motion field divergence...

[+] Stage 4: Transcribing speech & evaluating audio channel...


100%|███████████████████████████████████████| 139M/139M [00:02<00:00, 66.4MiB/s]



             PHYSICS & TEMPORAL VIDEO AUTHENTICATION REPORT

[1] HARDWARE & CONTAINER PROVENANCE
------------------------------------------------------------
╒══════════════════════╤═════════════════════════════════════════════════╕
│ Parameter            │ Audit Result                                    │
╞══════════════════════╪═════════════════════════════════════════════════╡
│ Camera Make/Model    │ None None                                       │
├──────────────────────┼─────────────────────────────────────────────────┤
│ Muxer / Encoder      │ Unknown                                         │
├──────────────────────┼─────────────────────────────────────────────────┤
│ Frame Rate           │ 24                                              │
├──────────────────────┼─────────────────────────────────────────────────┤
│ Duration             │ 0:00:30                                         │
├──────────────────────┼─────────────────────────────────────────────────┤
│ Identified Red

In [16]:
# ==============================================================================
# 1. INSTALL ZERO-COST SYSTEM & FORENSIC LIBRARIES
# ==============================================================================
!apt-get -qq update && apt-get -qq install -y exiftool ffmpeg

!pip install -q \
    opencv-python-headless \
    scenedetect[opencv] \
    numpy \
    scipy \
    openai-whisper \
    tabulate \
    Pillow

import os
import json
import subprocess
import cv2
import numpy as np
import scipy.fftpack as fftpack
import whisper
import torch
from google.colab import files
from tabulate import tabulate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[+] Operational Hardware: {DEVICE.upper()}")

# ==============================================================================
# 2. MEDIA INGESTION
# ==============================================================================
print("\n" + "=" * 60)
print("STAGE 1: MEDIA INGESTION")
print("=" * 60)
uploaded = files.upload()
if not uploaded:
    raise ValueError("No video file selected. Run cell again to upload.")

VIDEO_PATH = list(uploaded.keys())[0]
print(f"[+] Processing Target: {VIDEO_PATH}")

# ==============================================================================
# 3. CONTAINER & ENCODER FORENSICS (EXIFTOOL)
# ==============================================================================
def audit_container_internals(file_path):
    print("\n[+] Stage 2: Auditing deep container headers...")
    cmd = ["exiftool", "-j", file_path]
    res = subprocess.run(cmd, capture_output=True, text=True)
    try:
        exif = json.loads(res.stdout)[0]
    except Exception:
        exif = {}

    encoder = exif.get("Encoder", exif.get("CompressorName", "Unknown"))
    software = exif.get("Software", "None")
    make = exif.get("Make", "None")
    model = exif.get("Model", "None")

    # Flag indicators
    anomalies = []
    if make == "None" and model == "None":
        anomalies.append("Missing physical sensor/camera hardware profile")
    if any(synth in str(encoder).lower() for synth in ["lavf", "ffmpeg", "isom"]):
        anomalies.append("Synthetic/software muxer signature (Lavf/FFmpeg pipeline)")

    return {
        "Make/Model": f"{make} {model}".strip(),
        "Encoder": encoder,
        "Software": software,
        "FPS": exif.get("VideoFrameRate", "N/A"),
        "Duration": exif.get("Duration", "N/A"),
        "Anomalies": anomalies
    }

container_data = audit_container_internals(VIDEO_PATH)

# ==============================================================================
# 4. FREQUENCY DOMAIN (2D FFT) ARTIFACT ANALYSIS
# ==============================================================================
def analyze_fft_spectrum(frame_gray):
    """
    Computes 2D Fast Fourier Transform to find high-frequency checkerboard
    artifacts typical of generative diffusion/GAN upscalers.
    """
    # Compute 2D FFT and shift zero-frequency component to center
    f = np.fft.fft2(frame_gray)
    fshift = np.fft.fftshift(f)
    magnitude_spectrum = 20 * np.log(np.abs(fshift) + 1e-9)

    # Measure ratio of extreme high-frequency band vs low/mid frequencies
    h, w = frame_gray.shape
    cy, cx = h // 2, w // 2
    r_inner = min(h, w) // 8
    r_outer = min(h, w) // 4

    y, x = np.ogrid[:h, :w]
    mask_inner = (x - cx)**2 + (y - cy)**2 <= r_inner**2
    mask_outer = ((x - cx)**2 + (y - cy)**2 <= r_outer**2) & ~mask_inner

    high_freq_ratio = np.mean(magnitude_spectrum[mask_outer]) / (np.mean(magnitude_spectrum[mask_inner]) + 1e-6)
    return float(high_freq_ratio)

# ==============================================================================
# 5. TEMPORAL CONTINUITY & DENSE OPTICAL FLOW WARPING
# ==============================================================================
def inspect_temporal_and_motion(file_path, sample_limit=120):
    print("\n[+] Stage 3: Measuring temporal coherence & motion field divergence...")
    cap = cv2.VideoCapture(file_path)

    prev_gray = None
    flow_divergences = []
    fft_scores = []
    motion_magnitudes = []

    frame_idx = 0
    while cap.isOpened() and frame_idx < sample_limit:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 1. Spatial FFT analysis
        fft_ratio = analyze_fft_spectrum(gray)
        fft_scores.append(fft_ratio)

        # 2. Temporal Farneback dense optical flow
        if prev_gray is not None:
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, gray, None,
                pyr_scale=0.5, levels=3, winsize=15,
                iterations=3, poly_n=5, poly_sigma=1.2, flags=0
            )
            mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

            # Real camera footage has smooth, structured vector flow.
            # Generative video creates chaotic acceleration/warping vectors.
            flow_variance = np.var(mag)
            flow_divergences.append(flow_variance)
            motion_magnitudes.append(np.mean(mag))

        prev_gray = gray
        frame_idx += 1

    cap.release()

    avg_fft = float(np.mean(fft_scores)) if fft_scores else 0.0
    mean_flow_variance = float(np.mean(flow_divergences)) if flow_divergences else 0.0
    mean_motion = float(np.mean(motion_magnitudes)) if motion_magnitudes else 0.0

    return {
        "frames_analyzed": frame_idx,
        "avg_fft_high_freq_ratio": avg_fft,
        "mean_flow_variance": mean_flow_variance,
        "mean_motion_intensity": mean_motion
    }

motion_data = inspect_temporal_and_motion(VIDEO_PATH)

# ==============================================================================
# 6. AUDIO INTEGRITY & DIALOGUE CHECK (WHISPER - ZERO COST)
# ==============================================================================
def audit_audio_track(file_path):
    print("\n[+] Stage 4: Transcribing speech & evaluating audio channel...")
    whisper_model = whisper.load_model("base", device=DEVICE)
    res = whisper_model.transcribe(file_path, task="transcribe")
    return {
        "lang": res.get("language", "unknown").upper(),
        "transcript": res.get("text", "").strip()
    }

audio_data = audit_audio_track(VIDEO_PATH)

# ==============================================================================
# 7. FORENSIC SYNTHESIS & EXECUTIVE REPORT
# ==============================================================================
print("\n" + "=" * 76)
print("             PHYSICS & TEMPORAL VIDEO AUTHENTICATION REPORT")
print("=" * 76)

# Section 1: Container Audit
print("\n[1] HARDWARE & CONTAINER PROVENANCE")
print("-" * 60)
header_table = [
    ["Camera Make/Model", container_data["Make/Model"]],
    ["Muxer / Encoder", container_data["Encoder"]],
    ["Frame Rate", container_data["FPS"]],
    ["Duration", container_data["Duration"]],
    ["Identified Red Flags", "\n".join(container_data["Anomalies"]) if container_data["Anomalies"] else "None"]
]
print(tabulate(header_table, headers=["Parameter", "Audit Result"], tablefmt="fancy_grid"))

# Section 2: Motion & Frequency Physical Metrics
print("\n[2] PHYSICS-BASED TEMPORAL & SPECTRAL METRICS")
print("-" * 60)

# Decision logic based on physical properties
fft_val = motion_data["avg_fft_high_freq_ratio"]
flow_var = motion_data["mean_flow_variance"]

# High flow variance relative to overall motion indicates morphing/fluid drift
drift_ratio = flow_var / (motion_data["mean_motion_intensity"] + 1e-4)

metric_table = [
    ["Frames Sampled", motion_data["frames_analyzed"]],
    ["Spectral High-Frequency Ratio (FFT)", f"{fft_val:.3f} (Normal: 0.60 - 1.15)"],
    ["Dense Motion Flow Variance", f"{flow_var:.2f}"],
    ["Temporal Deformation / Drift Index", f"{drift_ratio:.2f} (Suspicious if > 15.0)"]
]
print(tabulate(metric_table, headers=["Forensic Metric", "Observed Value"], tablefmt="fancy_grid"))

# Formulate Verdict
is_synthetic = False
reasons = []

if drift_ratio > 15.0 and motion_data["mean_motion_intensity"] > 0.5:
    is_synthetic = True
    reasons.append("High non-rigid motion drift (unnatural pixel warping across consecutive frames)")

if fft_val > 1.25 or fft_val < 0.50:
    is_synthetic = True
    reasons.append("Anomalous frequency spectrum decay (typical of generative diffusion latents)")

if container_data["Anomalies"]:
    reasons.append("Complete absence of physical sensor noise and camera hardware headers")

print("\n[3] FORENSIC CONCLUSION")
print("-" * 60)
if is_synthetic:
    print("VERDICT: ⚠️ HIGH PROBABILITY OF GENERATIVE AI / SYNTHETIC VIDEO")
    print("Primary Indicators:")
    for r in reasons:
        print(f"  • {r}")
else:
    print("VERDICT: ✅ CONSISTENT WITH REAL-WORLD OPTICAL RECORDING")
    print("Physical metrics match coherent camera sensor motion and natural frequency decay.")

# Section 4: Spoken Track
print("\n[4] SPOKEN AUDIO CONTEXT")
print("-" * 60)
print(f"Language : {audio_data['lang']}")
print(f"Content  : {audio_data['transcript'] or '[No dialogue detected - Ambient/Muted]'}")
print("=" * 76)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[+] Operational Hardware: CUDA

STAGE 1: MEDIA INGESTION


Saving PXL_20250312_102828904.mp4 to PXL_20250312_102828904.mp4
[+] Processing Target: PXL_20250312_102828904.mp4

[+] Stage 2: Auditing deep container headers...

[+] Stage 3: Measuring temporal coherence & motion field divergence...

[+] Stage 4: Transcribing speech & evaluating audio channel...

             PHYSICS & TEMPORAL VIDEO AUTHENTICATION REPORT

[1] HARDWARE & CONTAINER PROVENANCE
------------------------------------------------------------
╒══════════════════════╤═════════════════════════════════════════════════╕
│ Parameter            │ Audit Result                                    │
╞══════════════════════╪═════════════════════════════════════════════════╡
│ Camera Make/Model    │ None None                                       │
├──────────────────────┼─────────────────────────────────────────────────┤
│ Muxer / Encoder      │ Unknown                                         │
├──────────────────────┼─────────────────────────────────────────────────┤
│ Frame Rate     

In [17]:
# ==============================================================================
# 1. INSTALL ZERO-COST SYSTEM & FORENSIC LIBRARIES
# ==============================================================================
!apt-get -qq update && apt-get -qq install -y exiftool ffmpeg

!pip install -q \
    opencv-python-headless \
    scenedetect[opencv] \
    numpy \
    scipy \
    openai-whisper \
    tabulate \
    Pillow

import os
import json
import subprocess
import cv2
import numpy as np
import scipy.fftpack as fftpack
import whisper
import torch
from google.colab import files
from tabulate import tabulate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[+] Operational Hardware: {DEVICE.upper()}")

# ==============================================================================
# 2. MEDIA INGESTION
# ==============================================================================
print("\n" + "=" * 60)
print("STAGE 1: MEDIA INGESTION")
print("=" * 60)
uploaded = files.upload()
if not uploaded:
    raise ValueError("No video file selected. Run cell again to upload.")

VIDEO_PATH = list(uploaded.keys())[0]
print(f"[+] Processing Target: {VIDEO_PATH}")

# ==============================================================================
# 3. CONTAINER & ENCODER FORENSICS (EXIFTOOL)
# ==============================================================================
def audit_container_internals(file_path):
    print("\n[+] Stage 2: Auditing deep container headers...")
    cmd = ["exiftool", "-j", file_path]
    res = subprocess.run(cmd, capture_output=True, text=True)
    try:
        exif = json.loads(res.stdout)[0]
    except Exception:
        exif = {}

    encoder = exif.get("Encoder", exif.get("CompressorName", "Unknown"))
    software = exif.get("Software", "None")
    make = exif.get("Make", "None")
    model = exif.get("Model", "None")

    # Flag indicators
    anomalies = []
    if make == "None" and model == "None":
        anomalies.append("Missing physical sensor/camera hardware profile")
    if any(synth in str(encoder).lower() for synth in ["lavf", "ffmpeg", "isom"]):
        anomalies.append("Synthetic/software muxer signature (Lavf/FFmpeg pipeline)")

    return {
        "Make/Model": f"{make} {model}".strip(),
        "Encoder": encoder,
        "Software": software,
        "FPS": exif.get("VideoFrameRate", "N/A"),
        "Duration": exif.get("Duration", "N/A"),
        "Anomalies": anomalies
    }

container_data = audit_container_internals(VIDEO_PATH)

# ==============================================================================
# 4. FREQUENCY DOMAIN (2D FFT) ARTIFACT ANALYSIS
# ==============================================================================
def analyze_fft_spectrum(frame_gray):
    """
    Computes 2D Fast Fourier Transform to find high-frequency checkerboard
    artifacts typical of generative diffusion/GAN upscalers.
    """
    # Compute 2D FFT and shift zero-frequency component to center
    f = np.fft.fft2(frame_gray)
    fshift = np.fft.fftshift(f)
    magnitude_spectrum = 20 * np.log(np.abs(fshift) + 1e-9)

    # Measure ratio of extreme high-frequency band vs low/mid frequencies
    h, w = frame_gray.shape
    cy, cx = h // 2, w // 2
    r_inner = min(h, w) // 8
    r_outer = min(h, w) // 4

    y, x = np.ogrid[:h, :w]
    mask_inner = (x - cx)**2 + (y - cy)**2 <= r_inner**2
    mask_outer = ((x - cx)**2 + (y - cy)**2 <= r_outer**2) & ~mask_inner

    high_freq_ratio = np.mean(magnitude_spectrum[mask_outer]) / (np.mean(magnitude_spectrum[mask_inner]) + 1e-6)
    return float(high_freq_ratio)

# ==============================================================================
# 5. TEMPORAL CONTINUITY & DENSE OPTICAL FLOW WARPING
# ==============================================================================
def inspect_temporal_and_motion(file_path, sample_limit=120):
    print("\n[+] Stage 3: Measuring temporal coherence & motion field divergence...")
    cap = cv2.VideoCapture(file_path)

    prev_gray = None
    flow_divergences = []
    fft_scores = []
    motion_magnitudes = []

    frame_idx = 0
    while cap.isOpened() and frame_idx < sample_limit:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 1. Spatial FFT analysis
        fft_ratio = analyze_fft_spectrum(gray)
        fft_scores.append(fft_ratio)

        # 2. Temporal Farneback dense optical flow
        if prev_gray is not None:
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, gray, None,
                pyr_scale=0.5, levels=3, winsize=15,
                iterations=3, poly_n=5, poly_sigma=1.2, flags=0
            )
            mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

            # Real camera footage has smooth, structured vector flow.
            # Generative video creates chaotic acceleration/warping vectors.
            flow_variance = np.var(mag)
            flow_divergences.append(flow_variance)
            motion_magnitudes.append(np.mean(mag))

        prev_gray = gray
        frame_idx += 1

    cap.release()

    avg_fft = float(np.mean(fft_scores)) if fft_scores else 0.0
    mean_flow_variance = float(np.mean(flow_divergences)) if flow_divergences else 0.0
    mean_motion = float(np.mean(motion_magnitudes)) if motion_magnitudes else 0.0

    return {
        "frames_analyzed": frame_idx,
        "avg_fft_high_freq_ratio": avg_fft,
        "mean_flow_variance": mean_flow_variance,
        "mean_motion_intensity": mean_motion
    }

motion_data = inspect_temporal_and_motion(VIDEO_PATH)

# ==============================================================================
# 6. AUDIO INTEGRITY & DIALOGUE CHECK (WHISPER - ZERO COST)
# ==============================================================================
def audit_audio_track(file_path):
    print("\n[+] Stage 4: Transcribing speech & evaluating audio channel...")
    whisper_model = whisper.load_model("base", device=DEVICE)
    res = whisper_model.transcribe(file_path, task="transcribe")
    return {
        "lang": res.get("language", "unknown").upper(),
        "transcript": res.get("text", "").strip()
    }

audio_data = audit_audio_track(VIDEO_PATH)

# ==============================================================================
# 7. FORENSIC SYNTHESIS & EXECUTIVE REPORT
# ==============================================================================
print("\n" + "=" * 76)
print("             PHYSICS & TEMPORAL VIDEO AUTHENTICATION REPORT")
print("=" * 76)

# Section 1: Container Audit
print("\n[1] HARDWARE & CONTAINER PROVENANCE")
print("-" * 60)
header_table = [
    ["Camera Make/Model", container_data["Make/Model"]],
    ["Muxer / Encoder", container_data["Encoder"]],
    ["Frame Rate", container_data["FPS"]],
    ["Duration", container_data["Duration"]],
    ["Identified Red Flags", "\n".join(container_data["Anomalies"]) if container_data["Anomalies"] else "None"]
]
print(tabulate(header_table, headers=["Parameter", "Audit Result"], tablefmt="fancy_grid"))

# Section 2: Motion & Frequency Physical Metrics
print("\n[2] PHYSICS-BASED TEMPORAL & SPECTRAL METRICS")
print("-" * 60)

# Decision logic based on physical properties
fft_val = motion_data["avg_fft_high_freq_ratio"]
flow_var = motion_data["mean_flow_variance"]

# High flow variance relative to overall motion indicates morphing/fluid drift
drift_ratio = flow_var / (motion_data["mean_motion_intensity"] + 1e-4)

metric_table = [
    ["Frames Sampled", motion_data["frames_analyzed"]],
    ["Spectral High-Frequency Ratio (FFT)", f"{fft_val:.3f} (Normal: 0.60 - 1.15)"],
    ["Dense Motion Flow Variance", f"{flow_var:.2f}"],
    ["Temporal Deformation / Drift Index", f"{drift_ratio:.2f} (Suspicious if > 15.0)"]
]
print(tabulate(metric_table, headers=["Forensic Metric", "Observed Value"], tablefmt="fancy_grid"))

# Formulate Verdict
is_synthetic = False
reasons = []

if drift_ratio > 15.0 and motion_data["mean_motion_intensity"] > 0.5:
    is_synthetic = True
    reasons.append("High non-rigid motion drift (unnatural pixel warping across consecutive frames)")

if fft_val > 1.25 or fft_val < 0.50:
    is_synthetic = True
    reasons.append("Anomalous frequency spectrum decay (typical of generative diffusion latents)")

if container_data["Anomalies"]:
    reasons.append("Complete absence of physical sensor noise and camera hardware headers")

print("\n[3] FORENSIC CONCLUSION")
print("-" * 60)
if is_synthetic:
    print("VERDICT: ⚠️ HIGH PROBABILITY OF GENERATIVE AI / SYNTHETIC VIDEO")
    print("Primary Indicators:")
    for r in reasons:
        print(f"  • {r}")
else:
    print("VERDICT: ✅ CONSISTENT WITH REAL-WORLD OPTICAL RECORDING")
    print("Physical metrics match coherent camera sensor motion and natural frequency decay.")

# Section 4: Spoken Track
print("\n[4] SPOKEN AUDIO CONTEXT")
print("-" * 60)
print(f"Language : {audio_data['lang']}")
print(f"Content  : {audio_data['transcript'] or '[No dialogue detected - Ambient/Muted]'}")
print("=" * 76)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[+] Operational Hardware: CUDA

STAGE 1: MEDIA INGESTION


Saving tester 3.mp4 to tester 3.mp4
[+] Processing Target: tester 3.mp4

[+] Stage 2: Auditing deep container headers...

[+] Stage 3: Measuring temporal coherence & motion field divergence...

[+] Stage 4: Transcribing speech & evaluating audio channel...

             PHYSICS & TEMPORAL VIDEO AUTHENTICATION REPORT

[1] HARDWARE & CONTAINER PROVENANCE
------------------------------------------------------------
╒══════════════════════╤═════════════════════════════════════════════════╕
│ Parameter            │ Audit Result                                    │
╞══════════════════════╪═════════════════════════════════════════════════╡
│ Camera Make/Model    │ None None                                       │
├──────────────────────┼─────────────────────────────────────────────────┤
│ Muxer / Encoder      │ Google                                          │
├──────────────────────┼─────────────────────────────────────────────────┤
│ Frame Rate           │ 23.976                            